# D2-02 · Two-stage detectors: anchors, matching, RPN, RoIAlign

### Steps
1. Anchors: a grid of reference boxes at several scales and aspect ratios
2. Matching anchors to ground truth (positive / negative / ignore) and the imbalance it creates
3. Box encoding (tx, ty, tw, th), decoding, smooth-L1
4. The Region Proposal Network: head, training on shapes, proposal filtering
5. RoI pooling vs RoIAlign: the quantisation problem and bilinear sampling
6. The second stage and the four-term Faster R-CNN loss
7. Fine-tune torchvision's Faster R-CNN on Penn-Fudan pedestrians and measure mAP

### Topics covered and learning objectives
- Generate anchors for any feature map and count them
- Implement the IoU-based matching rule and see why only ~1% of anchors are positive
- Encode / decode boxes relative to anchors and explain the log for width and height
- Build and train an RPN head; turn its dense output into a few hundred proposals
- Implement RoIAlign and verify it against torchvision
- Adapt a pretrained Faster R-CNN to a new dataset in ~15 lines

### Time estimate
~90 min in class (guided + hands-on)

### Prerequisites
D2-01 (boxes, IoU, NMS, mAP).

## 0 · Setup

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib, time
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from cvcourse import viz, boxes, shapes, tests, torchkit, day2, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
GREEN, RED, GREY, BLUE, ORANGE = "#54A24B", "#E45756", "#9a9a9a", "#4C78A8", "#F58518"

## The big picture

Faster R-CNN (Ren et al. 2015) is the model this notebook takes apart. One backbone computes
a feature map *once*; a small **Region Proposal Network** slides over it and scores a fixed
set of reference boxes (**anchors**) for "objectness" (the probability that the box holds *some*
object, whatever its class); the best few hundred become
**proposals**; **RoIAlign** cuts a fixed-size feature tile out of the map for each proposal;
a second head classifies each tile and refines its box. Two stages: *where might objects be?*
then *what exactly is there?*

In [ ]:
fig, ax = plt.subplots(figsize=(15, 3.6)); ax.axis("off"); ax.set_xlim(0, 15); ax.set_ylim(0, 3.6)
blocks = [(0.1, "image\n3 x H x W", "#dddddd"), (2.1, "backbone\n(ResNet + FPN)", "#9ecae1"), (4.3, "feature map\nC x H/16 x W/16", "#c6dbef"),
          (6.5, "RPN head\nobjectness A\ndeltas 4A per cell", ORANGE), (8.7, "proposals\ndecode, NMS,\ntop-N", "#fdd0a2"),
          (10.9, "RoIAlign\nN x C x 7 x 7", GREEN), (13.0, "box head\nclass + box\nper RoI", "#B279A2")]
for x, t, c in blocks:
    ax.add_patch(FancyBboxPatch((x, 1.0), 1.8, 1.8, boxstyle="round,pad=0.05", fc=c, ec="k", alpha=0.85))
    ax.text(x + 0.9, 1.9, t, ha="center", va="center", fontsize=9)
for x0, x1 in [(1.95, 2.1), (3.95, 4.3), (6.15, 6.5), (8.35, 8.7), (10.55, 10.9), (12.75, 13.0)]:
    ax.annotate("", (x1, 1.9), (x0, 1.9), arrowprops=dict(arrowstyle="-|>", lw=1.8))
ax.annotate("", (11.4, 2.85), (5.2, 2.85), arrowprops=dict(arrowstyle="-|>", lw=1.5, ls="--", color=GREY, connectionstyle="arc3,rad=-0.25"))
ax.text(8.3, 3.45, "the same feature map is cropped for every proposal", ha="center", fontsize=9, color="#555")
ax.text(7.6, 0.55, "stage 1: where? (class-agnostic)", ha="center", fontsize=10, color=ORANGE, weight="bold")
ax.text(12.9, 0.55, "stage 2: what? (per class)", ha="center", fontsize=10, color="#8a4f86", weight="bold")
plt.show()

## 1 · Anchors

A feature map with stride $s$ has one cell per $s \times s$ block of input pixels; cell
$(i, j)$ is centred at $\big((j + 0.5)s,\ (i + 0.5)s\big)$ in the image. At every cell we
place the same family of $A$ **anchors**: every combination of a few *scales* (box side for a
square anchor) and *aspect ratios* $r = h/w$, keeping the area fixed:

$$w = \frac{\text{scale}}{\sqrt r}, \qquad h = \text{scale} \cdot \sqrt r .$$

The network never predicts boxes from scratch - it predicts **corrections** to these anchors.
A feature map of $H_f \times W_f$ cells therefore produces $H_f W_f A$ candidate boxes.

### Exercise 1 · `make_anchors`

Return all anchors of one level as `(fh * fw * A, 4)` xyxy, ordered cell by cell (row, column,
anchor). Build the `A` base boxes centred at 0, the grid of centres, and add them with broadcasting.

<details class="api-ref" data-api="np.sqrt"><summary><b>Functions you will need</b> (click to open): <code>np.sqrt</code></summary>
<p><b><code>np.sqrt(x)</code></b><br>Square root of every element. Anchors use it to change the shape at a fixed area: for scale s and aspect ratio r = h / w, w = s / sqrt(r) and h = s * sqrt(r), so w * h = s^2 for every ratio.</p><pre>&gt;&gt;&gt; np.sqrt(np.array([4.0, 2.0, 0.25]))
array([2.        , 1.41421356, 0.5       ])
&gt;&gt;&gt; r = np.array([0.5, 1.0, 2.0]); w, h = 64 / np.sqrt(r), 64 * np.sqrt(r)
&gt;&gt;&gt; w.round(1), h.round(1), (w * h).round()     # three shapes, one area: 64 * 64
(array([90.5, 64. , 45.3]), array([45.3, 64. , 90.5]), array([4096., 4096., 4096.]))</pre><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.array,np.arange,np.stack,Tensor.reshape"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.array</code> · <code>np.arange</code> · <code>np.stack</code> · <code>Tensor.reshape</code></summary>
<ul><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>np.stack(arrays, axis=0)</code> &nbsp; Joins arrays of the same shape along a NEW axis: k arrays of shape (N,) give (k, N) with axis=0, or (N, k) with axis=-1. The box conversions use it to put four (N,) coordinate columns back together into one (N, 4) array.</li><li><code>x.reshape(*shape)</code> &nbsp; Returns the same elements in a new shape, read in row order (the last dimension changes fastest). One size may be -1: it is computed from the others. NumPy arrays have the same method.</li></ul></details>

In [ ]:
def make_anchors(feature_size, stride, scales, ratios):
    '''feature_size = (fh, fw) cells, stride in pixels, scales and ratios: lists ->
    all anchors of the level, (fh * fw * A, 4) xyxy, A = len(scales) * len(ratios),
    ordered cell by cell (row, then column), then anchor by anchor.'''
    fh, fw = feature_size
    # 1. the A base boxes, centred at (0, 0): one per (scale, ratio) pair
    base = []
    for scale in scales:
        for ratio in ratios:
            # YOUR CODE: w and h of this anchor: w = scale / sqrt(ratio), h = scale * sqrt(ratio)
            # YOUR CODE: append the box [-w / 2, -h / 2, w / 2, h / 2] to base
            raise NotImplementedError
    base = np.array(base)                            # (A, 4)
    # 2. the centre of every cell, row by row, in pixels
    centres = []
    for i in range(fh):                              # row
        for j in range(fw):                          # column
            # YOUR CODE: cx = (j + 0.5) * stride, cy = (i + 0.5) * stride; append [cx, cy, cx, cy]
            raise NotImplementedError
    centres = np.array(centres)                      # (fh * fw, 4)
    # 3. YOUR CODE: every centre plus every base box, with broadcasting:
    #               centres[:, None, :] + base[None, :, :] -> (fh * fw, A, 4)
    # 4. YOUR CODE: reshape to (fh * fw * A, 4) and return it
    raise NotImplementedError

tests.check_make_anchors(make_anchors)

<details class="api-ref" data-api="torchkit.get_penn_fudan,F.interpolate,Tensor.permute,Tensor.numpy,np.array"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_penn_fudan</code> · <code>F.interpolate</code> · <code>Tensor.permute</code> · <code>Tensor.numpy</code> · <code>np.array</code></summary>
<ul><li><code>pf = torchkit.get_penn_fudan()</code> &nbsp; Course helper. Returns the Penn-Fudan pedestrian dataset (170 street photos), downloading it into data/ if needed. An item is (image, target): a float image (3, H, W) in [0, 1] at its original size, and a dict in the torchvision detection format: &quot;boxes&quot; (N, 4) as x1, y1, x2, y2 pixels, &quot;labels&quot; (N,) all 1 (= person), &quot;masks&quot; (N, H, W) of 0 / 1, one per person, plus &quot;image_id&quot;, &quot;area&quot; and &quot;iscrowd&quot;.</li><li><code>F.interpolate(x, size=None, scale_factor=None, mode=&quot;nearest&quot;, align_corners=None)</code> &nbsp; Resizes a batch of maps (N, C, H, W) to size=(h, w), or by scale_factor. mode=&quot;nearest&quot; repeats pixels, mode=&quot;bilinear&quot; blends neighbouring pixels for a smooth result (with align_corners=False, the usual convention for images). E.g. to upsample a coarse 5 x 5 or 7 x 7 Grad-CAM map to the image size.</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li></ul></details>

In [ ]:
from PIL import Image
pf = torchkit.get_penn_fudan()
img0 = pf[3][0]
s = 512 / max(img0.shape[-2:])
img0 = F.interpolate(img0[None], scale_factor=s, mode="bilinear", antialias=True)[0].permute(1, 2, 0).numpy()
H, W = img0.shape[:2]
scales, ratios = [64, 128, 256], [0.5, 1.0, 2.0]
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
ax = axes[0]; ax.imshow(img0); ax.set_title("one cell (stride 32): 3 scales x 3 ratios = 9 anchors")
cell = (6, 8)                                   # (row, col)
fam = boxes.make_anchors((1, 1), 32, scales, ratios) + [cell[1] * 32, cell[0] * 32] * 2
for k, b in enumerate(fam):
    ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=[RED, ORANGE, BLUE][k // 3], lw=2, ls=["-", "--", ":"][k % 3]))
ax.plot(*((np.array(cell[::-1]) + 0.5) * 32), "w*", ms=14, mec="k")
ax.set_xlim(-150, W + 150); ax.set_ylim(H + 150, -150)
for c, sc in zip([RED, ORANGE, BLUE], scales):
    ax.plot([], [], color=c, lw=2, label=f"scale {sc}")
for ls, r in zip(["-", "--", ":"], ratios):
    ax.plot([], [], color="k", ls=ls, label=f"ratio h/w = {r}")
ax.legend(loc="lower right", fontsize=8)
ax = axes[1]; ax.imshow(img0)
fh, fw = H // 32, W // 32
grid = boxes.make_anchors((fh, fw), 32, [64], [1.0])
ctr = (grid[:, :2] + grid[:, 2:]) / 2
ax.plot(ctr[:, 0], ctr[:, 1], ".", color="yellow", ms=4)
for x in range(0, W + 1, 32): ax.axvline(x, color="white", lw=0.4, alpha=0.6)
for y in range(0, H + 1, 32): ax.axhline(y, color="white", lw=0.4, alpha=0.6)
for k in range(0, len(grid), 37):
    b = grid[k]; ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=ORANGE, lw=1.2))
ax.set_title(f"stride 32 -> {fh} x {fw} cells; every cell gets the same 9 anchors (a few 64 px ones drawn)")
ax.set_xlim(0, W); ax.set_ylim(H, 0)
plt.tight_layout(); plt.show()

print(f"{'stride':>7} {'feature map':>12} {'anchors (A=9)':>14}")
tot = 0
for st in [8, 16, 32]:
    n = (512 // st) ** 2 * 9; tot += n
    print(f"{st:>7} {f'{512 // st} x {512 // st}':>12} {n:>14,d}")
print(f"{'total':>7} {'':>12} {tot:>14,d}   for a 512 x 512 image - and maybe 5 of them contain an object")

Left: the family of one cell - colour is the scale, line style the ratio; all nine share the
centre (star). Right: that family is repeated at every cell of the grid. The table is the
point to remember: anchors grow with $1/s^2$, so the fine stride-8 level alone holds three
quarters of the ~48 000 candidates.

## 2 · Matching anchors to ground truth

Training needs a target for every anchor. Faster R-CNN's RPN rule, from the IoU matrix
`anchors x GT`:

* **positive** (label 1) if its best IoU with any GT is $\ge 0.7$; it regresses to that GT;
* **negative** (label 0, background) if its best IoU is $< 0.3$;
* **ignore** (label -1) in between - ambiguous anchors are simply left out of the loss;
* **low-quality matches**: every GT also makes its single best anchor positive, even if that IoU
  is below 0.7 - otherwise small or oddly shaped objects would get no positive anchor at all.

0.7 / 0.3 are the paper's thresholds, chosen for ~800 px images. They are arguments of the
function, and further down you will see why our 128 px images need 0.5 / 0.3 instead.

### Lab · Anchor matching

**Predict first:** two objects in a 256 px image, stride 16, 3 scales x 3 ratios = 2304 anchors,
the paper's 0.7 / 0.3 rule. How many anchors are positive: about 6, 60 or 600?
Check in the lab, then work through its "Try this" list (8-10 minutes).

In [ ]:
labs.show("anchor-matching")

### Exercise 2 · `match_anchors`

Return `(matched_gt_index, labels)` for `anchors (N, 4)` and `gt (G, 4)`: two integer arrays of
length `N`, labels in {1, 0, -1}; `matched_gt_index` only matters where `labels == 1`. Use
`boxes.box_iou`. The green anchors in the lab are the `labels == 1` entries; the dashed "forced"
ones come from the `allow_low_quality` branch.

<details class="api-ref" data-api="np.full,Tensor.argmax,Tensor.max,np.random.default_rng,Tensor.sum,Tensor.mean"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.full</code> · <code>Tensor.argmax</code> · <code>Tensor.max</code> · <code>np.random.default_rng</code> · <code>Tensor.sum</code> · <code>Tensor.mean</code></summary>
<ul><li><code>np.full(shape, value)</code> &nbsp; An array of the given shape (a number or a tuple) with every element set to value.</li><li><code>x.argmax(dim)</code> &nbsp; The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li><li><code>x.mean(dim=None, keepdim=False)</code> &nbsp; The average of all elements, or along dim (a number or a tuple), which then disappears from the shape. For a batch (B, C, H, W), mean(dim=(2, 3)) averages over the pixels and gives (B, C): global average pooling. NumPy arrays have the same method, with axis= for dim=.</li></ul></details>

In [ ]:
def match_anchors(anchors, gt, pos_iou=0.7, neg_iou=0.3, allow_low_quality=True):
    '''anchors (N, 4), gt (G, 4), xyxy -> (best_gt, labels): two integer arrays of length N.
    labels: 1 positive, 0 negative (background), -1 ignore; best_gt[n] = the GT that anchor n regresses to.'''
    # 1. the IoU of every anchor with every GT
    ious = boxes.box_iou(anchors, gt)                 # (N, G)
    # 2. for every anchor: its best GT, and that IoU
    best_gt = ious.argmax(1)                          # (N,) index of the best GT (along axis 1 = over the GTs)
    best_iou = ious.max(1)                            # (N,) its IoU
    # 3. start with every anchor on "ignore"
    labels = np.full(len(anchors), -1)                # (N,)
    # 4. YOUR CODE: negatives: label 0 where best_iou < neg_iou (a boolean mask as the index)
    # 5. YOUR CODE: positives: label 1 where best_iou >= pos_iou
    # 6. low-quality rule: every GT also makes its best anchor positive
    if allow_low_quality:
        # YOUR CODE: best_anchor, the best anchor of every GT: argmax over the anchors (axis 0) -> (G,)
        # YOUR CODE: label those anchors 1
        # YOUR CODE: and point them at their GT: best_gt[best_anchor] = np.arange(len(gt))
        raise NotImplementedError
    return best_gt, labels

tests.check_match_anchors(match_anchors)

In [ ]:
# Matching on a shapes image: 128 px, stride 8 -> 16 x 16 cells, 3 scales x 3 ratios = 2304 anchors
m_img, m_gt, m_lab, _ = shapes.make_sample(np.random.default_rng(7), 128)
anchors = boxes.make_anchors((16, 16), 8, [16, 32, 48], [0.5, 1.0, 2.0])
for thr in (0.7, 0.5):
    print(f"positive threshold {thr}: {(boxes.match_anchors(anchors, m_gt, thr, 0.3)[1] == 1).sum()} positive anchors for {len(m_gt)} objects")
idx, lab = boxes.match_anchors(anchors, m_gt, 0.5, 0.3)        # 0.5 for our tiny images (see below)
max_iou = boxes.box_iou(anchors, m_gt).max(1)
ctr = (anchors[:, :2] + anchors[:, 2:]) / 2

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2), gridspec_kw=dict(width_ratios=[1, 1, 1.25]))
axes[0].imshow(m_img); axes[0].set_title(f"positive anchors ({(lab == 1).sum()}), coloured by their GT")
for b, g in zip(anchors[lab == 1], idx[lab == 1]):
    axes[0].add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=viz.color(g + 2), lw=1.2))
viz.draw_boxes(axes[0], m_gt, edgecolor="k", lw=2.5)
axes[1].imshow(m_img); axes[1].set_title(f"anchor centres: ignore ({(lab == -1).sum()}) / negative ({(lab == 0).sum()})")
ig = lab == -1
axes[1].scatter(ctr[lab == 0, 0], ctr[lab == 0, 1], s=3, c=RED, alpha=0.3)
for b in anchors[ig]:
    axes[1].add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=GREY, lw=0.6, alpha=0.6))
for ax in axes[:2]: ax.set_xlim(0, 128); ax.set_ylim(128, 0); ax.axis("off")
axes[2].hist([max_iou[lab == 0], max_iou[lab == -1], max_iou[lab == 1]], bins=30, stacked=True,
             color=[RED, GREY, GREEN], label=["negative", "ignore", "positive"], log=True)
axes[2].axvline(0.3, color="k", ls="--"); axes[2].axvline(0.5, color="k", ls="--"); axes[2].axvline(0.7, color=GREY, ls=":")
axes[2].set_xlabel("max IoU of each anchor with any GT"); axes[2].set_ylabel("anchors (log scale)"); axes[2].legend()
axes[2].set_title(f"{100 * (lab == 1).mean():.1f}% positive, {100 * (lab == 0).mean():.0f}% negative")
plt.tight_layout(); plt.show()

With the 0.7 rule the four objects get only 5 positive anchors: two objects reach 0.7 with one
or two anchors, the other two only through the forced "best anchor per GT" rule (the dashed
anchors of the lab). On a 128 px image a stride-8 anchor can be 4 px off-centre, and 4 px on a
24 px object already pushes IoU below 0.7.
Faster R-CNN's thresholds assume ~800 px images where objects are large relative to the stride;
for our tiny images we use **0.5 / 0.3** (RetinaNet uses 0.5 / 0.4). Anchor design and matching
thresholds are dataset-dependent hyper-parameters - one of the motivations for anchor-free detectors.

The histogram is the core difficulty of dense detection: **almost every anchor is
background**. If the objectness loss averaged over all anchors, "always say background"
would score 99% accuracy. The RPN therefore **samples** a minibatch of 256 anchors per image
with at most 50% positives (Faster R-CNN; torchvision's default) - a sampling cousin of
hard-negative mining. One-stage detectors cannot afford to throw anchors away; their answer
is focal loss (D2-03).

## 3 · Encoding boxes relative to anchors

A positive anchor $a$ learns to regress its GT $g$. Instead of raw coordinates the target is
a scale-free **delta**:

$$t_x = \frac{g_x - a_x}{a_w},\quad t_y = \frac{g_y - a_y}{a_h},\quad t_w = \log\frac{g_w}{a_w},\quad t_h = \log\frac{g_h}{a_h}$$

(centres and sizes, i.e. cxcywh). Dividing by the anchor size makes a "half an anchor to the
right" mean the same for small and large anchors. The **log** makes size changes symmetric
(twice as big = $+0.69$, half as big = $-0.69$) and keeps the predicted size positive after
decoding with $\exp$. Decoding is the exact inverse.

### Lab · Box encoding

**Predict first:** take the same anchor and ground truth and double both (the same scene at twice
the resolution). Which of $t_x, t_y, t_w, t_h$ change? Check with the "x2 scale both boxes"
button, then work through the "Try this" list (6-8 minutes).

In [ ]:
labs.show("box-encoding")

### Exercise 3 · `encode` / `decode`

Both take aligned `(N, 4)` xyxy arrays and return `(N, 4)` arrays (`encode`: deltas
$(t_x, t_y, t_w, t_h)$, `decode`: xyxy boxes). Convert with `boxes.xyxy_to_cxcywh` /
`boxes.cxcywh_to_xyxy`. The checker tests what the lab's "snap deltas to the target" button
showed: `decode(encode(g, a), a)` gives back `g`.

<details class="api-ref" data-api="np.log,np.exp"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.log</code> · <code>np.exp</code></summary>
<ul><li><code>np.log(x)</code> &nbsp; The natural logarithm (base e) of every element. ln C is the cross-entropy loss of a classifier that guesses uniformly among C classes: ln 37 = 3.61.</li><li><code>np.exp(x)</code> &nbsp; e = 2.718... to the power of every element. Soft-NMS multiplies scores by exp(-IoU^2 / sigma): 1 for no overlap, close to 0 for a heavy duplicate. It also decodes log-encoded box sizes (D2-02).</li></ul></details>

In [ ]:
def encode(gt, anchors):
    '''gt, anchors: (N, 4) xyxy, aligned (gt[n] belongs to anchors[n]) -> deltas (N, 4) = (tx, ty, tw, th).'''
    # 1. both in cxcywh: columns 0, 1, 2, 3 are cx, cy, w, h
    g = boxes.xyxy_to_cxcywh(gt)                     # (N, 4)
    a = boxes.xyxy_to_cxcywh(anchors)                # (N, 4)
    # 2. YOUR CODE: tx, ty, the centre shift in units of the anchor's size:
    #               (g_cx - a_cx) / a_w and (g_cy - a_cy) / a_h -> (N,) each
    # 3. YOUR CODE: tw, th, the log of the size ratio: log(g_w / a_w) and log(g_h / a_h) -> (N,) each
    # 4. YOUR CODE: put the columns tx, ty, tw, th together with np.stack([...], -1) -> (N, 4); return it
    raise NotImplementedError

def decode(deltas, anchors):
    '''deltas (N, 4) = (tx, ty, tw, th), anchors (N, 4) xyxy -> boxes (N, 4) xyxy; the inverse of encode.'''
    # 1. the anchors in cxcywh
    a = boxes.xyxy_to_cxcywh(anchors)                # (N, 4)
    # 2. YOUR CODE: cx, cy, the centre: the anchor's centre plus tx (ty) times the anchor's width (height)
    # 3. YOUR CODE: w, h, the size: the anchor's width (height) times exp(tw) (exp(th))
    # 4. YOUR CODE: stack cx, cy, w, h -> (N, 4), convert with boxes.cxcywh_to_xyxy and return it
    raise NotImplementedError

tests.check_encode_decode(encode, decode)

<details class="api-ref" data-api="np.allclose"><summary><b>Functions used here</b> (click to open): <code>np.allclose</code></summary>
<p><b><code>np.allclose(a, b, rtol=1e-05, atol=1e-08)</code></b><br>True if every element of a is close to the matching one of b (|a - b| &lt;= atol + rtol * |b|), as one bool. The right way to compare floats, which rarely match exactly after a computation and its inverse.</p><pre>&gt;&gt;&gt; x = np.array([0.1, 0.2]) * 3
&gt;&gt;&gt; x == np.array([0.3, 0.6]), np.allclose(x, [0.3, 0.6])
(array([False, False]), True)</pre><p>See also: <code>np.array_equal</code>, <code>torch.allclose</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.where,np.linspace,np.abs"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.where</code> · <code>np.linspace</code> · <code>np.abs</code></summary>
<ul><li><code>np.where(condition)  or  np.where(condition, x, y)</code> &nbsp; With one argument: the indices where condition is True, as a tuple with one index array per dimension (take [0] for a 1-D array). With three: a new array that takes x where condition is True and y elsewhere, element by element.</li><li><code>np.linspace(start, stop, num=50)</code> &nbsp; num evenly spaced numbers from start to stop, both ends included. Used for plot axes and grids.</li><li><code>np.abs(a)</code> &nbsp; The absolute value of every element. np.abs(a - b).max() is the largest difference between two arrays: the course&#x27;s check that two implementations compute the same numbers (up to tiny rounding differences).</li></ul></details>

In [ ]:
k = np.where(lab == 1)[0][len(np.where(lab == 1)[0]) // 3]
a, g = anchors[k], m_gt[idx[k]]
d = boxes.encode_boxes(g[None], a[None])[0]
ac, gc = boxes.xyxy_to_cxcywh(a), boxes.xyxy_to_cxcywh(g)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw=dict(width_ratios=[1.1, 1, 1]))
ax = axes[0]; ax.imshow(m_img)
ax.add_patch(Rectangle(a[:2], *(a[2:] - a[:2]), fill=False, ec=ORANGE, lw=2.5, ls="--", label="anchor"))
ax.add_patch(Rectangle(g[:2], *(g[2:] - g[:2]), fill=False, ec=GREEN, lw=2.5, label="GT"))
ax.annotate("", gc[:2], ac[:2], arrowprops=dict(arrowstyle="->", lw=2, color="k"))
ax.plot(*ac[:2], "o", color=ORANGE); ax.plot(*gc[:2], "o", color=GREEN)
pad = 20
ax.set_xlim(min(a[0], g[0]) - pad, max(a[2], g[2]) + pad); ax.set_ylim(max(a[3], g[3]) + pad, min(a[1], g[1]) - pad)
ax.legend(loc="upper right", fontsize=8)
ax.set_title(f"tx={d[0]:+.2f}  ty={d[1]:+.2f}  tw={d[2]:+.2f}  th={d[3]:+.2f}", fontsize=10)
r = np.linspace(0.25, 4, 200)
axes[1].plot(r, np.log(r), color=BLUE, lw=2, label="t_w = log(g_w / a_w)")
axes[1].plot(r, r - 1, color=GREY, ls="--", label="naive (g_w - a_w) / a_w")
for x in [0.5, 2]: axes[1].plot(x, np.log(x), "o", color=RED)
axes[1].text(2.05, np.log(2) - 0.35, "x2: +0.69", color=RED); axes[1].text(0.55, np.log(0.5) - 0.3, "/2: -0.69", color=RED)
axes[1].set_xscale("log"); axes[1].set_xticks([0.25, 0.5, 1, 2, 4]); axes[1].set_xticklabels(["1/4", "1/2", "1", "2", "4"])
axes[1].xaxis.set_minor_formatter(plt.NullFormatter()); axes[1].set_xlabel("size ratio GT / anchor (log axis)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.3)
axes[1].set_title("log encoding: symmetric for growing and shrinking")
x = np.linspace(-1.5, 1.5, 400)
axes[2].plot(x, np.abs(x), color=GREY, label="L1"); axes[2].plot(x, 0.5 * x ** 2, color=ORANGE, label="L2 / 2")
for beta, c in [(1 / 9, BLUE), (1.0, GREEN)]:
    axes[2].plot(x, boxes.smooth_l1(x, beta), color=c, lw=2, label=f"smooth-L1, beta={beta:.2g}")
axes[2].set_ylim(0, 1.3); axes[2].legend(fontsize=8); axes[2].grid(alpha=.3); axes[2].set_xlabel("prediction - target")
axes[2].set_title("smooth-L1: quadratic near 0, linear for outliers")
plt.tight_layout(); plt.show()
print("round trip decode(encode(g, a), a) == g:", np.allclose(boxes.decode_boxes(d[None], a[None])[0], g))

Left: the arrow is the centre shift that $t_x, t_y$ encode, in units of the anchor's size (deltas in
the title). Middle: on a log axis, growing x2 and shrinking /2 give targets of equal size
with opposite signs; the naive ratio (grey) does not.

**Smooth-L1** (Huber) is the box loss of the R-CNN family: quadratic for errors below $\beta$
(smooth gradient near the optimum), linear above (a badly matched anchor cannot dominate the
batch the way it would with L2). Faster R-CNN's RPN uses $\beta = 1/9$.

## 4 · The Region Proposal Network

The RPN head is tiny: a $3\times 3$ conv (256 or 512 channels) + ReLU shared by two sibling
$1\times 1$ convs. At every cell it outputs $A$ **objectness logits** ("is anchor $a$ of this cell
on an object?") and $4A$ **deltas**. It is class-agnostic: it does not know about people or
cats, only "thing vs stuff".

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.4)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(-0.6, 4.2)
def slab(x, y, w, h, d, c):
    for k in range(d - 1, -1, -1):
        ax.add_patch(Rectangle((x + 0.08 * k, y + 0.08 * k), w, h, fc=c, ec="k", lw=0.7))
slab(0.3, 1.0, 1.8, 1.8, 5, "#c6dbef"); ax.text(1.35, 0.55, "feature map\n(N, C, 32, 32)", ha="center", va="top", fontsize=9)
slab(4.0, 1.0, 1.8, 1.8, 5, "#fdd0a2"); ax.text(5.05, 0.55, "3x3 conv + ReLU\n(N, C, 32, 32)", ha="center", va="top", fontsize=9)
slab(8.4, 2.5, 1.2, 1.0, 3, ORANGE)
slab(8.4, 0.0, 1.2, 1.0, 8, "#9ecae1")
ax.text(10.3, 3.0, "objectness logits (N, A, 32, 32)\n-> (N, 32*32*A)", fontsize=9, va="center")
ax.text(10.3, 0.7, "box deltas (N, 4A, 32, 32)\n-> (N, 32*32*A, 4)", fontsize=9, va="center")
ax.annotate("", (4.0, 1.9), (2.5, 1.9), arrowprops=dict(arrowstyle="-|>", lw=1.8))
ax.annotate("", (8.35, 3.0), (6.3, 2.3), arrowprops=dict(arrowstyle="-|>", lw=1.5)); ax.text(6.9, 2.95, "1x1 conv, A", fontsize=9)
ax.annotate("", (8.35, 0.5), (6.3, 1.5), arrowprops=dict(arrowstyle="-|>", lw=1.5)); ax.text(6.9, 0.65, "1x1 conv, 4A", fontsize=9)
plt.show()

### Exercise 4 · `RPNHead`

`RPNHead(in_channels, num_anchors)`; `forward(x)` returns `(objectness, deltas)` with shapes
`(N, A, H, W)` and `(N, 4A, H, W)`. Name the layers `conv`, `cls_logits`, `bbox_pred` (like torchvision).

<details class="api-ref" data-api="nn.Conv2d,F.relu"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.Conv2d</code> · <code>F.relu</code></summary>
<ul><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>F.relu(x)</code> &nbsp; The rectified linear unit, element by element: negative values become 0, the others pass unchanged. The function form, for use inside forward(); nn.ReLU() is the same operation as a layer, for nn.Sequential.</li></ul></details>

In [ ]:
class RPNHead(nn.Module):
    '''RPNHead(C, A): input feature map x (N, C, H, W) ->
    (objectness logits (N, A, H, W), box deltas (N, 4A, H, W)): A scores and 4A deltas at every cell.'''
    def __init__(self, in_channels, num_anchors):
        super().__init__()
        # 1. YOUR CODE: self.conv, a 3x3 convolution from in_channels to in_channels, padding=1 (same size)
        # 2. YOUR CODE: self.cls_logits, a 1x1 convolution from in_channels to num_anchors (one score per anchor)
        # 3. YOUR CODE: self.bbox_pred, a 1x1 convolution from in_channels to 4 * num_anchors (4 deltas per anchor)
        raise NotImplementedError

    def forward(self, x):
        # 4. YOUR CODE: h, the shared features: self.conv, then F.relu -> (N, C, H, W)
        # 5. YOUR CODE: objectness = self.cls_logits(h) -> (N, A, H, W)
        # 6. YOUR CODE: deltas = self.bbox_pred(h) -> (N, 4A, H, W)
        # 7. YOUR CODE: return objectness, deltas
        raise NotImplementedError

tests.check_rpn_head(RPNHead)

### Train a tiny RPN on shapes (about a minute on CPU)

`day2.TinyRPN` = a 5-layer backbone with stride 8 (128 px -> 16 x 16 cells) + **your** head
(or the reference one if your checker failed), 9 anchors per cell. Targets come from
`match_anchors` (0.5 / 0.3) + `encode_boxes`, computed once for the whole dataset. Each step samples 128
anchors per image (<= 25% positives), and the loss is BCE on the sampled anchors' objectness +
smooth-L1 on the positive anchors' deltas: exactly Faster R-CNN's RPN loss.

<details class="api-ref" data-api="torch.sigmoid"><summary><b>Functions used here</b> (click to open): <code>torch.sigmoid</code></summary>
<p><b><code>torch.sigmoid(x)</code></b><br>Squashes every value (a logit) into (0, 1): sigmoid(x) = 1 / (1 + exp(-x)). Turns an objectness logit into a probability; D2-03 and D2-04 also use it to keep predicted box coordinates between 0 and 1.</p><pre>&gt;&gt;&gt; torch.sigmoid(torch.tensor([-4.6, 0.0, 2.0]))
tensor([0.0100, 0.5000, 0.8808])</pre><p><i>Watch out:</i> Losses such as F.binary_cross_entropy_with_logits take the raw logits: do not apply sigmoid first.</p><p>See also: <code>F.binary_cross_entropy_with_logits</code>, <code>Tensor.softmax</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Module.state_dict,Module.load_state_dict,Tensor.float,torch.no_grad,Tensor.view"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Module.state_dict</code> · <code>Module.load_state_dict</code> · <code>Tensor.float</code> · <code>torch.no_grad</code> · <code>Tensor.view</code></summary>
<ul><li><code>model.state_dict()</code> &nbsp; An ordered dict from names to tensors holding everything that defines the model&#x27;s state: all parameters (weights, biases) and buffers (BatchNorm&#x27;s running_mean and running_var). Names follow the module structure, e.g. &quot;layer1.0.conv1.weight&quot;. Optimizers and schedulers have a state_dict() too (momentum, Adam&#x27;s moment estimates, the step counter): to resume training, save all three.</li><li><code>model.load_state_dict(state_dict, strict=True)</code> &nbsp; Copies the tensors of a state_dict into an existing model&#x27;s parameters and buffers, matched by name; names and shapes must agree. Returns a report of missing and unexpected names. Optimizers and schedulers have load_state_dict too.</li><li><code>x.float()</code> &nbsp; Returns the tensor converted to float32, the dtype layers and losses work in (the same as x.to(torch.float32)); x itself is not changed. A uint8 image becomes 0.0 to 255.0, so divide by 255 to get [0, 1].</li><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li><li><code>x.view(*shape)</code> &nbsp; Like reshape, but guaranteed not to copy: the result shares memory with x. One size may be -1. The course uses v.view(-1, 1, 1) to turn C per-channel values into shape (C, 1, 1), which broadcasts against (C, H, W) images; indexing with None, v[:, None, None], does the same.</li></ul></details>

In [ ]:
head_cls = RPNHead if tests.check_rpn_head(RPNHead, verbose=False) else day2.RPNHead
print("using", "your RPNHead" if head_cls is RPNHead else "the reference day2.RPNHead")
torch.manual_seed(0)
rpn = day2.TinyRPN(head_cls, channels=64, scales=(16, 32, 48), ratios=(0.5, 1.0, 2.0))
X_tr, B_tr, _ = day2.shapes_tensors(1000, 128, seed=0)
X_va, B_va, _ = day2.shapes_tensors(100, 128, seed=1)
anc_np = rpn.anchors.numpy()
lab_tr, dt_tr = day2.rpn_targets(anc_np, B_tr)
print(f"{len(anc_np)} anchors per image; positives per image: {(lab_tr == 1).sum(1).float().mean():.1f} on average")

untrained = day2.TinyRPN(head_cls); untrained.load_state_dict(rpn.state_dict())   # keep a copy for the "before" picture
t0 = time.time()
rpn_hist = day2.train_tiny_rpn(rpn, X_tr, lab_tr, dt_tr, epochs=6, batch_size=32, lr=3e-3, max_seconds=150)
print(f"trained in {time.time() - t0:.0f} s")

In [ ]:
# Objectness map and proposals before / after training, on validation images
fig, axes = plt.subplots(3, 4, figsize=(15, 11.5))
for c, i in enumerate([0, 3, 7, 11]):
    img = X_va[i]
    with torch.no_grad():
        obj, _ = rpn(img[None])
    objmap = torch.sigmoid(obj[0]).view(16, 16, 9).max(-1).values.numpy()
    p_before, s_before = day2.rpn_proposals(untrained, img, post_nms_top_n=10)
    p_after, s_after = day2.rpn_proposals(rpn, img, post_nms_top_n=10)
    im = img.permute(1, 2, 0).numpy()
    axes[0, c].imshow(im); viz.draw_boxes(axes[0, c], p_before.numpy(), edgecolor=RED, lw=1); axes[0, c].set_title("before training: top-10 proposals", fontsize=9)
    axes[1, c].imshow(im); axes[1, c].imshow(objmap, cmap="viridis", alpha=0.6, extent=(0, 128, 128, 0), vmin=0, vmax=1)
    axes[1, c].set_title("objectness (max over 9 anchors)", fontsize=9)
    axes[2, c].imshow(im); viz.draw_boxes(axes[2, c], B_va[i], edgecolor=GREEN, lw=3, alpha=0.6)
    viz.draw_boxes(axes[2, c], p_after.numpy()[:5], scores=s_after.numpy()[:5], edgecolor=BLUE, lw=1.5)
    axes[2, c].set_title("after: top-5 proposals (GT green)", fontsize=9)
for ax in axes.flat: ax.axis("off")
plt.tight_layout(); plt.show()

Top row: before training the highest-scoring anchors are arbitrary boxes. Middle: after training
the objectness map is bright only on the shapes, background cells stay dark. Bottom: the
best-scoring decoded proposals now hug the green ground truth, although no anchor matched it
exactly - the deltas moved them there. A few lower-scoring duplicates are still off; NMS and the
second stage deal with those.

### From 2304 scored anchors to a handful of proposals

The RPN's raw output is one decoded box per anchor. torchvision's `RegionProposalNetwork`
turns it into proposals with a fixed recipe (default numbers in brackets for a full-size model):

1. decode all deltas against their anchors; 2. clip to the image and drop tiny boxes;
3. keep the top-k by objectness *per level* (2000 train / 1000 test); 4. NMS at IoU 0.7;
5. keep the top-N (2000 train / 1000 test). The second stage only ever sees these.

<details class="api-ref" data-api="Tensor.argsort"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.argsort</code></summary>
<ul><li><code>x.argsort(dim=-1, descending=False)</code> &nbsp; The method form of torch.argsort: the indices that would sort the tensor along dim. x.argsort(descending=True)[:k] are the positions of the k largest values, the same as torch.topk(x, k).indices.</li></ul></details>

In [ ]:
i = 3
_, _, steps = day2.rpn_proposals(rpn, X_va[i], pre_nms_top_n=100, post_nms_top_n=12, nms_iou=0.5, return_steps=True)
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
titles = {"all": "all decoded anchors (top 300 shown)", "clipped": "clipped, tiny ones removed",
          "top-k": "top-100 by objectness", "nms": "after NMS: top-12 proposals"}
for ax, (k, (bx, sc)) in zip(axes, steps.items()):
    o = sc.argsort(descending=True)[:300]
    ax.imshow(X_va[i].permute(1, 2, 0).numpy())
    cols = plt.cm.viridis(sc[o].numpy())
    for b, c in zip(bx[o].numpy()[::-1], cols[::-1]):
        ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec=c, lw=0.8))
    ax.set_title(f"{titles[k]}\n{len(bx)} boxes", fontsize=9); ax.set_xlim(-10, 138); ax.set_ylim(138, -10); ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · RoI pooling vs RoIAlign

The second stage needs a **fixed-size** feature tile (7 x 7) for every proposal, whatever its
size. Fast R-CNN's **RoI pooling** snaps the proposal to the feature grid (round the box
coordinates), splits it into 7 x 7 *integer* bins (rounding again) and max-pools each bin.
With stride 16, one rounding step can move the box by up to 8 image pixels - harmless for
classification, harmful for precise boxes and fatal for masks.

**RoIAlign** (Mask R-CNN, 2017) never rounds: the box stays in continuous coordinates, each
bin gets a few regularly spaced **sampling points** (2 x 2), the feature value at each point is
obtained by **bilinear interpolation** of its 4 neighbouring cells, and the samples are averaged.

<details class="api-ref" data-api="torchvision.ops.roi_align,np.add.outer,np.sin,np.cos"><summary><b>Functions used here</b> (click to open): <code>torchvision.ops.roi_align</code> · <code>np.add.outer</code> · <code>np.sin</code> · <code>np.cos</code></summary>
<p><b><code>torchvision.ops.roi_align(input, boxes, output_size, spatial_scale=1.0, sampling_ratio=-1, aligned=False)</code></b><br>Cuts a fixed-size P x P tile (P = output_size) out of a feature map for every box, by bilinear sampling at exact, unrounded positions and averaging sampling_ratio x sampling_ratio samples per bin. input is (N, C, H, W); boxes is a list with one (L, 4) xyxy tensor per image, in image pixels; spatial_scale = 1 / stride converts them to feature-map cells. The output is (K, C, P, P) for K boxes in total.</p><pre>&gt;&gt;&gt; feat = torch.arange(16.).reshape(1, 1, 4, 4)       # one image, one channel, 4 x 4
&gt;&gt;&gt; feat[0, 0]
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])
&gt;&gt;&gt; rois = [torch.tensor([[0., 0., 4., 4.]])]          # one box: the whole map
&gt;&gt;&gt; torchvision.ops.roi_align(feat, rois, 2, spatial_scale=1.0, sampling_ratio=2, aligned=True)
tensor([[[[ 2.5000,  4.5000],
          [10.5000, 12.5000]]]])</pre><p><i>Watch out:</i> aligned=True treats pixel centres as sitting at +0.5 (Detectron2, Exercise 5); the default False is a legacy version shifted by half a pixel, which torchvision&#x27;s own detectors still use.</p><p>See also: <code>torchvision.ops.box_iou</code></p><p><b><code>np.add.outer(a, b)</code></b><br>All pairwise sums: out[i, j] = a[i] + b[j], with shape (len(a), len(b)); the same as a[:, None] + b[None, :]. D2-02 uses it to build small test feature maps with a visible pattern.</p><pre>&gt;&gt;&gt; np.add.outer(np.arange(3), np.array([0, 10, 20, 30]))
array([[ 0, 10, 20, 30],
       [ 1, 11, 21, 31],
       [ 2, 12, 22, 32]])</pre><p><b><code>np.sin(x)</code></b><br>Sine of every element, x in radians (np.pi = 180 degrees). D2-02 builds a smooth test feature map from sines and cosines of the row and column index.</p><pre>&gt;&gt;&gt; np.sin(np.array([0.0, np.pi / 6, np.pi / 2]))
array([0. , 0.5, 1. ])</pre><p>See also: <code>np.cos</code></p><p><b><code>np.cos(x)</code></b><br>Cosine of every element, x in radians (np.pi = 180 degrees).</p><pre>&gt;&gt;&gt; np.cos(np.array([0.0, np.pi / 3, np.pi]))
array([ 1. ,  0.5, -1. ])</pre><p>See also: <code>np.sin</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.tensor"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.tensor</code></summary>
<ul><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li></ul></details>

In [ ]:
fmap = torch.tensor(np.add.outer(np.arange(8), np.arange(8)) % 5 + np.random.default_rng(0).uniform(0, 1, (8, 8)), dtype=torch.float32)[None]
roi = np.array([1.3, 1.7, 6.1, 5.6])              # in feature-map coordinates (stride 1 for the drawing)
pooled, bins = day2.roi_pool_quantized(fmap, roi, 3)
aligned = torchvision.ops.roi_align(fmap[None], [torch.tensor(roi[None], dtype=torch.float32)], 3, 1.0, 2, aligned=True)[0, 0]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6), gridspec_kw=dict(width_ratios=[1, 0.45, 1, 0.45]))
for ax in (axes[0], axes[2]):
    ax.imshow(fmap[0], cmap="Blues", extent=(0, 8, 8, 0), vmin=0, vmax=6)
    ax.set_xticks(range(9)); ax.set_yticks(range(9)); ax.grid(color="k", lw=0.5)
    ax.add_patch(Rectangle(roi[:2], *(roi[2:] - roi[:2]), fill=False, ec=RED, lw=2.5, ls="--"))
for (xa, ya, xb, yb) in bins:
    axes[0].add_patch(Rectangle((xa, ya), xb - xa, yb - ya, fill=False, ec=ORANGE, lw=2.2))
axes[0].set_title("RoI pooling: box rounded to the grid,\ninteger bins (orange) - some cells used twice", fontsize=10)
bw, bh = (roi[2] - roi[0]) / 3, (roi[3] - roi[1]) / 3
for i in range(3):
    for j in range(3):
        axes[2].add_patch(Rectangle((roi[0] + j * bw, roi[1] + i * bh), bw, bh, fill=False, ec=GREEN, lw=1.8))
        for sy in range(2):
            for sx in range(2):
                axes[2].plot(roi[0] + (j + (sx + 0.5) / 2) * bw, roi[1] + (i + (sy + 0.5) / 2) * bh, "o", color=RED, ms=5)
axes[2].set_title("RoIAlign: exact bins (green), 2 x 2 bilinear\nsamples per bin (red dots), averaged", fontsize=10)
for ax, m, t in [(axes[1], pooled[0], "RoI pool 3x3"), (axes[3], aligned, "RoIAlign 3x3")]:
    ax.imshow(m, cmap="Blues", vmin=0, vmax=6); viz.annotate_grid(ax, m.numpy(), fmt="{:.2f}"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Quantisation in action: move the RoI by fractions of a feature cell and watch one output bin
shifts = np.linspace(0, 2, 81)
pool_v, align_v = [], []
smooth = torch.tensor(np.add.outer(np.sin(np.arange(16) / 2.0), np.cos(np.arange(16) / 3.0)), dtype=torch.float32)[None]
for sft in shifts:
    r = np.array([3.0 + sft, 4.0, 9.0 + sft, 10.0])
    pool_v.append(float(day2.roi_pool_quantized(smooth, r, 3)[0][0, 1, 1]))
    align_v.append(float(torchvision.ops.roi_align(smooth[None], [torch.tensor(r[None], dtype=torch.float32)], 3, 1.0, 2, aligned=True)[0, 0, 1, 1]))
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(shifts, pool_v, color=ORANGE, lw=2, drawstyle="steps-mid", label="RoI pooling (rounds the box)")
ax.plot(shifts, align_v, color=GREEN, lw=2, label="RoIAlign (bilinear)")
ax.set_xlabel("sub-cell shift of the RoI (feature cells; x16 = image pixels)"); ax.set_ylabel("centre bin value")
ax.legend(); ax.grid(alpha=.3); ax.set_title("RoI pooling is piecewise constant in the box position: no gradient, jumps")
plt.tight_layout(); plt.show()

### Lab · RoI pooling vs RoIAlign

**Predict first:** a RoIAlign sample point lies a quarter of a cell to the right of and a quarter
of a cell below the centre of a feature cell. What are its four bilinear weights? Hover a sample
in the lab to check, then work through the "Try this" list (8 minutes).

In [ ]:
labs.show("roi-align")

### Exercise 5 · RoIAlign in PyTorch

Two functions. `bilinear(feature, y, x)` interpolates a `(C, H, W)` map at float coordinates
`y`, `x` (tensors of the same shape) and returns `(C, *y.shape)` (the clamping is given).
`roi_align(feature, rois, output_size, spatial_scale, sampling_ratio)` returns `(K, C, P, P)`
and loops over RoIs: scale the box to feature coordinates, shift by -0.5 (`aligned=True`: pixel *centres*),
build the `P*s` sample coordinates along y and along x, sample the full grid, and average the
`s x s` samples of every bin. These are exactly the numbers in the lab's calculation box: the
"-0.5 -> index" line, then four neighbours times their weights. The checker compares with
`torchvision.ops.roi_align`.

<details class="api-ref" data-api="Tensor.floor,Tensor.long,Tensor.new_zeros,torch.meshgrid"><summary><b>Functions you will need</b> (click to open): <code>Tensor.floor</code> · <code>Tensor.long</code> · <code>Tensor.new_zeros</code> · <code>torch.meshgrid</code></summary>
<p><b><code>x.floor()</code></b><br>Rounds every element DOWN to a whole number (towards minus infinity); the result is still a float tensor. In bilinear sampling, y.floor() is the index of the grid row at or just above a sample point, and y - y.floor() the sample&#x27;s fractional distance below that row.</p><pre>&gt;&gt;&gt; y = torch.tensor([1.75, 2.0, -0.25])
&gt;&gt;&gt; y.floor()
tensor([ 1.,  2., -1.])
&gt;&gt;&gt; y - y.floor()                   # fractional part, always in [0, 1)
tensor([0.7500, 0.0000, 0.7500])</pre><p>See also: <code>Tensor.long</code>, <code>Tensor.round</code></p><p><b><code>x.long()</code></b><br>Converts to int64 (torch.long), the dtype PyTorch wants for indices and class labels. Floats are cut towards zero (1.7 -&gt; 1, -1.7 -&gt; -1), so floor first when values can be negative.</p><pre>&gt;&gt;&gt; x = torch.tensor([1.7, -1.7, 3.0])
&gt;&gt;&gt; x.long()
tensor([ 1, -1,  3])
&gt;&gt;&gt; x.floor().long()
tensor([ 1, -2,  3])</pre><p>See also: <code>Tensor.floor</code>, <code>Tensor.float</code>, <code>Tensor.to</code></p><p><b><code>x.new_zeros(size)</code></b><br>A tensor of zeros of the given size with the SAME dtype and device as x. Handy inside a function: the output buffer is on the GPU whenever the input is, without passing device= around.</p><pre>&gt;&gt;&gt; feat = torch.ones(3, 8, 8, dtype=torch.float64)
&gt;&gt;&gt; out = feat.new_zeros((2, 3, 4, 4))
&gt;&gt;&gt; out.shape, out.dtype
(torch.Size([2, 3, 4, 4]), torch.float64)</pre><p>See also: <code>torch.zeros</code>, <code>torch.zeros_like</code></p><p><b><code>torch.meshgrid(ys, xs, indexing=&#x27;ij&#x27;)</code></b><br>From a 1-D tensor of row coordinates (length H) and one of column coordinates (length W), builds two (H, W) tensors that together list every (row, column) point of the grid. With indexing=&quot;ij&quot; the first output changes down the rows, the second along the columns. RoIAlign uses it for its sample points.</p><pre>&gt;&gt;&gt; yy, xx = torch.meshgrid(torch.tensor([0., 1.]), torch.tensor([10., 20., 30.]), indexing=&quot;ij&quot;)
&gt;&gt;&gt; yy
tensor([[0., 0., 0.],
        [1., 1., 1.]])
&gt;&gt;&gt; xx
tensor([[10., 20., 30.],
        [10., 20., 30.]])</pre><p><i>Watch out:</i> Always pass indexing: np.meshgrid defaults to &quot;xy&quot; (x first, outputs of shape (len(y), len(x))), torch.meshgrid to &quot;ij&quot;; mixing them up transposes the grid.</p><p>See also: <code>np.meshgrid</code>, <code>np.mgrid</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Tensor.clamp,Tensor.tolist,torch.arange"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Tensor.clamp</code> · <code>Tensor.tolist</code> · <code>torch.arange</code></summary>
<ul><li><code>x.clamp(min=None, max=None)</code> &nbsp; Limits every value to [min, max]: smaller values become min, larger ones max; either bound can be left out. Used to keep an image in [0, 1] before plotting it (plt.imshow clips and warns otherwise).</li><li><code>x.tolist()</code> &nbsp; Converts a tensor into (nested) Python lists of plain Python numbers, for printing a few values readably. A 0-dimensional tensor gives a single number, like .item().</li><li><code>torch.arange(end) / torch.arange(start, end, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a 1-D tensor: from start (default 0) up to but not including end. Integer arguments give an int64 tensor; a float argument or dtype=torch.float32 gives floats. With reshape, the quickest way to build a small tensor whose values are easy to follow.</li></ul></details>

In [ ]:
def bilinear(feature, y, x):
    '''feature (C, H, W); y, x: float tensors of one same shape S (row and column of each point) ->
    (C, *S): the feature map interpolated at every point (y, x).'''
    C, H, W = feature.shape
    # 1. keep the points inside the map
    y = y.clamp(0, H - 1)
    x = x.clamp(0, W - 1)
    # 2. the four neighbouring cells of every point: rows y0 (above) and y1 (below),
    #    columns x0 (left) and x1 (right), as integer indices
    y0 = y.floor().long()
    x0 = x.floor().long()
    y1 = (y0 + 1).clamp(max=H - 1)
    x1 = (x0 + 1).clamp(max=W - 1)
    ly = y - y0                                  # in [0, 1): how far the point lies below row y0
    lx = x - x0                                  # in [0, 1): how far it lies right of column x0
    # 3. YOUR CODE: top_left, feature[:, y0, x0] times its weight (1 - ly) * (1 - lx) -> (C, *S)
    # 4. YOUR CODE: top_right, feature[:, y0, x1] times (1 - ly) * lx
    # 5. YOUR CODE: bottom_left, feature[:, y1, x0] times ly * (1 - lx)
    # 6. YOUR CODE: bottom_right, feature[:, y1, x1] times ly * lx
    # 7. YOUR CODE: return the sum of the four (the four weights add up to 1)
    raise NotImplementedError

def roi_align(feature, rois, output_size=7, spatial_scale=1.0, sampling_ratio=2):
    '''feature (C, H, W), rois (K, 4) xyxy in image pixels -> (K, C, P, P), P = output_size:
    a P x P tile for every RoI, each bin the mean of sampling_ratio x sampling_ratio bilinear samples.'''
    C = feature.shape[0]
    P = output_size                                  # bins per side
    s = sampling_ratio                               # samples per bin and side
    out = feature.new_zeros((len(rois), C, P, P))    # (K, C, P, P), zeros of the same dtype as feature
    for k, box in enumerate(rois):
        # 1. the box in feature-map coordinates, shifted by -0.5 (aligned: pixel centres)
        x1, y1, x2, y2 = (box * spatial_scale - 0.5).tolist()
        bw = (x2 - x1) / P                           # width of one bin, in feature cells
        bh = (y2 - y1) / P                           # height of one bin
        # 2. where the s samples sit inside one bin, as a fraction of the bin: e.g. [0.25, 0.75] for s = 2
        offsets = (torch.arange(s, dtype=feature.dtype) + 0.5) / s     # (s,)
        bins = torch.arange(P, dtype=feature.dtype)                    # (P,) bin numbers 0 .. P-1
        # 3. YOUR CODE: ys, the P*s sample rows: y1 + (bin number + offset) * bh, for every bin and
        #               every offset: broadcast bins[:, None] against offsets[None, :] -> (P, s),
        #               then reshape(-1) -> (P*s,)
        # 4. YOUR CODE: xs, the P*s sample columns, the same with x1 and bw -> (P*s,)
        # 5. YOUR CODE: yy, xx, the full grid of sample points: torch.meshgrid(ys, xs, indexing="ij")
        #               -> (P*s, P*s) each
        # 6. YOUR CODE: vals, the feature at every sample point: bilinear(feature, yy, xx) -> (C, P*s, P*s)
        # 7. YOUR CODE: for every bin (i, j), i, j = 0 .. P-1: its s x s samples are
        #               vals[:, i*s:(i+1)*s, j*s:(j+1)*s] -> (C, s, s); out[k, :, i, j] = their mean over the
        #               last two axes -> (C,)
        raise NotImplementedError
    return out

tests.check_roi_align(roi_align)

## 6 · The second stage and the four-term loss

Each RoI tile (256 x 7 x 7) goes through a small head (two FC layers of 1024 in the original,
four convs + one FC in torchvision's `_v2`) and two sibling outputs: **class logits** over
$K + 1$ classes (background included) and **class-specific box deltas** $4(K+1)$ - a refinement
*of the proposal*, encoded exactly like the RPN's deltas relative to anchors. RoIs are matched to
GT with IoU $\ge 0.5$ (foreground) / $< 0.5$ (background) and sampled 512 per image with 25% foreground.

| term | stage | on which samples | loss |
|---|---|---|---|
| `loss_objectness` | RPN | 256 sampled anchors / image | binary cross-entropy |
| `loss_rpn_box_reg` | RPN | positive anchors | smooth-L1 on $(t_x, t_y, t_w, t_h)$ |
| `loss_classifier` | RoI head | 512 sampled RoIs / image | cross-entropy over $K+1$ classes |
| `loss_box_reg` | RoI head | foreground RoIs, GT class only | smooth-L1 on deltas vs the proposal |

These four names are exactly the keys of the dict torchvision returns in training mode - you
will see them in the training log below.

## 7 · Fine-tune torchvision's Faster R-CNN on Penn-Fudan

Penn-Fudan has 170 street images with 423 pedestrians (instance masks too - Day 3). The
recipe that works for almost any small detection dataset:

1. load a COCO-pretrained detector (here `fasterrcnn_resnet50_fpn_v2`, 43 M parameters);
2. **replace the box predictor** so it outputs your number of classes (+1 for background);
3. fine-tune everything with SGD, a small LR and a warm-up.

**CPU budget.** We train on **40 images** at **320 px** (`min_size=320`), with fewer proposals
(`rpn_post_nms_top_n_train=300`, `box_batch_size_per_image=64`), for 2 epochs (40 iterations)
capped at 3 minutes, and save `data/checkpoints/d2_02_frcnn_pennfudan.pt` so re-runs just load it.
The provided checkpoint was made by exactly this cell without the time cap (about 10 CPU minutes);
delete it to train yourself. With a GPU and the standard recipe (120 / 50 split, 800 px, 2-10
epochs) you get mAP@0.5 of ~0.97-0.99 and mAP@[.5:.95] of ~0.75-0.85.

In [ ]:
pf_train, pf_val = day2.penn_fudan_splits(n_train=40, n_val=20, seed=0, max_side=512)
print(f"Penn-Fudan subset: {len(pf_train)} train / {len(pf_val)} val images, "
      f"{sum(len(t['boxes']) for _, t in pf_train)} / {sum(len(t['boxes']) for _, t in pf_val)} people")
fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, i in zip(axes, range(5)):
    im, t = pf_train[i]
    ax.imshow(im.permute(1, 2, 0)); viz.draw_boxes(ax, t["boxes"].numpy(), edgecolor=GREEN, lw=2); ax.axis("off")
plt.tight_layout(); plt.show()

Before training anything: the COCO model already detects people (COCO class 1 = person, and
torchvision also uses label 1 for "person" in our dataset). How good is it out of the box?

<details class="api-ref" data-api="torchvision.models.detection.fasterrcnn_resnet50_fpn_v2"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.detection.fasterrcnn_resnet50_fpn_v2</code></summary>
<p><b><code>fasterrcnn_resnet50_fpn_v2(weights=None, num_classes=None, min_size=800, max_size=1333, ...)</code></b><br>Builds torchvision&#x27;s Faster R-CNN (ResNet-50 + FPN, RPN, RoIAlign, box head; improved &quot;v2&quot; recipe, 43.7 M parameters). It takes a LIST of (3, H, W) float images in [0, 1] and resizes them itself (min_size, max_size). Train mode needs targets, a list of dicts with &quot;boxes&quot; (K, 4) xyxy pixels and &quot;labels&quot; (K,) int64 starting at 1 (0 means background), and returns a dict of the four losses; eval mode returns one dict per image with &quot;boxes&quot;, &quot;labels&quot;, &quot;scores&quot;.</p><pre>&gt;&gt;&gt; from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
&gt;&gt;&gt; m = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, num_classes=2, min_size=128, max_size=128)
&gt;&gt;&gt; imgs = [torch.rand(3, 128, 128)]
&gt;&gt;&gt; targets = [{&quot;boxes&quot;: torch.tensor([[20., 30., 90., 100.]]), &quot;labels&quot;: torch.tensor([1])}]
&gt;&gt;&gt; sorted(m.train()(imgs, targets))        # train mode: the losses
[&#x27;loss_box_reg&#x27;, &#x27;loss_classifier&#x27;, &#x27;loss_objectness&#x27;, &#x27;loss_rpn_box_reg&#x27;]
&gt;&gt;&gt; sorted(m.eval()(imgs)[0])               # eval mode: detections of image 0
[&#x27;boxes&#x27;, &#x27;labels&#x27;, &#x27;scores&#x27;]</pre><p><i>Watch out:</i> weights=&quot;DEFAULT&quot; downloads the COCO weights (91 classes including background, about 170 MB, once); weights=None gives a random model. num_classes counts the background: a person detector has 2.</p><p>See also: <code>torchvision.models.detection.faster_rcnn.FastRCNNPredictor</code>, <code>torchvision.models.detection.fcos_resnet50_fpn</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="Module.eval"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>Module.eval</code></summary>
<ul><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li></ul></details>

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
FAST = dict(min_size=320, max_size=512, rpn_pre_nms_top_n_test=500, rpn_post_nms_top_n_test=100,
            rpn_pre_nms_top_n_train=1000, rpn_post_nms_top_n_train=300, box_batch_size_per_image=64)
coco_model = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT", **FAST).eval()
t0 = time.time()
coco_res = day2.evaluate_map(coco_model, pf_val)             # one inference pass; label 1 = person in both
print(f"COCO-pretrained, no fine-tuning: mAP@0.5 = {coco_res['map50']:.3f}, mAP@[.5:.95] = {coco_res['map']:.3f}  ({time.time() - t0:.0f} s)")
del coco_model

### Exercise 6 · replace the box predictor

`model.roi_heads.box_predictor` is a `FastRCNNPredictor(in_features, num_classes)` with
`cls_score` and `bbox_pred` linear layers. Replace it with a fresh one for **2 classes**
(background + person). Everything else keeps its COCO weights.

<details class="api-ref" data-api="torchvision.models.detection.faster_rcnn.FastRCNNPredictor"><summary><b>Functions you will need</b> (click to open): <code>torchvision.models.detection.faster_rcnn.FastRCNNPredictor</code></summary>
<p><b><code>FastRCNNPredictor(in_channels, num_classes)</code></b><br>The last layer of Faster R-CNN&#x27;s box head: two linear layers on each RoI&#x27;s feature vector, cls_score (num_classes logits) and bbox_pred (4 * num_classes deltas, one box per class). Installing a new one as model.roi_heads.box_predictor adapts a pretrained detector to your classes; num_classes counts the background, in_channels is the length of the RoI feature vector (cls_score.in_features of the old one).</p><pre>&gt;&gt;&gt; from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
&gt;&gt;&gt; head = FastRCNNPredictor(1024, 2)                   # background + 1 class
&gt;&gt;&gt; scores, deltas = head(torch.zeros(5, 1024))         # features of 5 RoIs
&gt;&gt;&gt; scores.shape, deltas.shape
(torch.Size([5, 2]), torch.Size([5, 8]))</pre><p>See also: <code>torchvision.models.detection.fasterrcnn_resnet50_fpn_v2</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

In [ ]:
torch.manual_seed(0)
model = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT", **FAST)
# 1. the old predictor (COCO: 91 classes) and the number of features its cls_score layer reads
old_predictor = model.roi_heads.box_predictor
# 2. YOUR CODE: in_features, the input size of old_predictor.cls_score (a Linear layer: .in_features)
# 3. YOUR CODE: install a new FastRCNNPredictor(in_features, 2) as model.roi_heads.box_predictor
#               (2 classes: background + person)
raise NotImplementedError

tests.check_box_predictor(model, num_classes=2)

<details class="api-ref" data-api="np.convolve"><summary><b>Functions used here</b> (click to open): <code>np.convolve</code></summary>
<p><b><code>np.convolve(a, v, mode=&#x27;full&#x27;)</code></b><br>Slides v along a and sums the products at every position (a 1-D convolution). With v = np.ones(k) / k it is a moving average over k values; mode=&quot;valid&quot; keeps only the positions where v fits entirely, so len(a) - k + 1 values come out. D2-02 smooths noisy loss curves this way.</p><pre>&gt;&gt;&gt; loss = np.array([4., 2., 3., 1., 2., 0.])
&gt;&gt;&gt; np.convolve(loss, np.ones(3) / 3, &quot;valid&quot;)      # mean of every window of 3
array([3., 2., 2., 1.])</pre><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchkit.count_parameters,np.ones"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.count_parameters</code> · <code>np.ones</code></summary>
<ul><li><code>torchkit.count_parameters(model, trainable_only=False)</code> &nbsp; Course helper. The total number of parameters of a model, sum(p.numel() for p in model.parameters()); trainable_only=True counts only those with requires_grad (useful once layers are frozen for fine-tuning).</li><li><code>np.ones(shape)</code> &nbsp; An array of the given shape (a number or a tuple) filled with 1.0, e.g. a constant learning-rate schedule: learning rate / peak = 1 at every step.</li></ul></details>

In [ ]:
print(f"parameters: {torchkit.count_parameters(model) / 1e6:.1f} M")

def train():
    # only called when there is no saved checkpoint yet
    return day2.train_detector(model, pf_train, epochs=2, batch_size=2, lr=0.01, max_minutes=3, log_every=5)

history = day2.load_or_train(model, "d2_02_frcnn_pennfudan.pt", train)
model.eval()
if history:
    keys = ["loss_objectness", "loss_rpn_box_reg", "loss_classifier", "loss_box_reg"]
    fig, ax = plt.subplots(figsize=(10, 3.6))
    for k, c in zip(keys, [ORANGE, "#fdae6b", "#B279A2", GREEN]):
        v = np.array([h[k] for h in history]); ax.plot(v, color=c, alpha=0.35)
        ax.plot(np.convolve(v, np.ones(5) / 5, "valid"), color=c, lw=2, label=k)
    ax.set_xlabel("iteration (batch of 2 images)"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=.3)
    ax.set_title("the four Faster R-CNN loss terms during fine-tuning (thin: raw, thick: moving average)")
    plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
ft_res = day2.evaluate_map(model, pf_val)
ft_preds, pf_gts = ft_res["preds"], ft_res["gts"]
print(f"{'':28s}{'mAP@0.5':>9}{'mAP@[.5:.95]':>14}")
print(f"{'COCO weights, no training':28s}{coco_res['map50']:9.3f}{coco_res['map']:14.3f}")
print(f"{'fine-tuned (40 imgs, CPU)':28s}{ft_res['map50']:9.3f}{ft_res['map']:14.3f}   (eval {time.time() - t0:.0f} s)")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, range(8)):
    im = pf_val[i][0].permute(1, 2, 0).numpy(); ax.imshow(im); ax.axis("off")
    viz.draw_boxes(ax, pf_gts[i]["boxes"], edgecolor=GREEN, lw=3, alpha=0.5)
    p = ft_preds[i]; k = p["scores"] > 0.5
    viz.draw_boxes(ax, p["boxes"][k], scores=p["scores"][k], edgecolor=RED, lw=1.5)
fig.suptitle("fine-tuned Faster R-CNN on validation images (green: GT, red: predictions with score > 0.5)", y=1.0)
plt.tight_layout(); plt.show()

**Read the table carefully.** With the provided checkpoint you should see roughly mAP@0.5 = 0.99
and mAP@[.5:.95] = 0.78 after fine-tuning - but the *untouched* COCO model already scores about
0.99 / 0.85. Persons are COCO's most frequent class, learned from 64 000 images at 800 px; our
40 images at 320 px with a freshly initialised predictor match its recall and lose some box
precision. Lessons: (1) **always measure the zero-shot / pretrained baseline first**; (2) fine-tuning
pays off when your classes or your box definition differ from the pretraining data (see the Pet
heads in D2-04); (3) on 20 validation images a difference of a few points is noise.

Then look at the failures: partly occluded people at the image border, groups where two people
merge into one box (NMS!), and small background pedestrians that were not annotated - a
"false positive" that is actually a labelling gap. Inspecting errors like this is worth more
than another decimal of mAP.

## 8 · Summary

* **Anchors** = a fixed family of scales x ratios at every feature-map cell; a 512 px image
  easily gets 50 000 of them. Networks predict *corrections* to anchors.
* **Matching**: IoU >= 0.7 positive, < 0.3 negative, in between ignored, plus the best anchor of
  every GT (0.5 / 0.3 for our 128 px images). ~1% of anchors are positive, hence minibatch sampling
  (two-stage) or focal loss (one-stage).
* **Encoding**: $t_x, t_y$ = centre offset / anchor size, $t_w, t_h$ = log size ratio; smooth-L1.
* **RPN**: 3x3 conv + two 1x1 convs (A objectness, 4A deltas); proposals = decode, clip, top-k, NMS, top-N.
* **RoIAlign** samples bilinearly at exact positions; RoI pooling rounds twice - the box edges (up to
  half a stride) and then the bin borders (up to about one more cell) - and its output jumps instead
  of following the box.
* **Faster R-CNN loss** = RPN objectness + RPN box + RoI classification + RoI box.
* Fine-tuning a library detector = replace the predictor for $K+1$ classes and train with a small LR.

### Self-check
1. How many anchors does a 640 x 480 image get from a single stride-16 level with 3 scales and 3 ratios?
2. An anchor has IoU 0.55 with its best GT. What is its label under the 0.7 / 0.3 rule? When could it still be positive?
3. Why encode the width as $\log(g_w/a_w)$ rather than $g_w - a_w$?
4. The RPN predicts objectness for 2 classes or for 81? Why?
5. A proposal spans x = 33.7 ... 97.2 px on a stride-16 map. Which x-range does RoI pooling actually use? RoIAlign?
6. Which of the four loss terms would you watch if proposals look good but final boxes are badly placed?

### Go deeper
* Ren, He, Girshick, Sun, *Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks* (NeurIPS 2015).
* He, Gkioxari, Dollar, Girshick, *Mask R-CNN* (ICCV 2017) - section 3 introduces RoIAlign.
* torchvision "TorchVision Object Detection Finetuning Tutorial" (Penn-Fudan, the recipe used here) and the source of `torchvision/models/detection/rpn.py`.